<!-- cspell:ignore arctan atleast cabibbo dalitz fromkeys keepdims linalg maksymowicz mystnb xscale yscale -->

# Phase space generators

:::{autolink-concat}
:::

{class}`.TFPhaseSpaceGenerator` and {class}`.TFWeightedPhaseSpaceGenerator` generate phase space samples with [`phasespace`](https://phasespace.readthedocs.io), which can run on NumPy, TensorFlow, or JAX. Despite their names, these generators use NumPy by default and do not require TensorFlow. This page compares the backends, so that you can decide which one fits your use case. All of them return a {obj}`.DataSample` of NumPy arrays with the same structure, so they can be swapped without changing the rest of an analysis.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
set_matplotlib_formats("svg")

## Available generators

The backend is selected with the `backend` argument of the generator. The mapping below is the single place that defines which backends this page compares. All tables and figures on this page loop over it.

In [ ]:
from tensorwaves.data import (
    NumpyUniformRNG,
    TFPhaseSpaceGenerator,
    TFUniformRealNumberGenerator,
)

initial_state_mass = 3.0969
final_state_masses = {0: 0.1350, 1: 0.1396, 2: 0.4937}
generators = {
    "numpy": (
        TFPhaseSpaceGenerator(initial_state_mass, final_state_masses, backend="numpy"),
        NumpyUniformRNG(seed=0),
    ),
    "tensorflow": (
        TFPhaseSpaceGenerator(
            initial_state_mass, final_state_masses, backend="tensorflow"
        ),
        TFUniformRealNumberGenerator(seed=0),
    ),
    "jax": (
        TFPhaseSpaceGenerator(initial_state_mass, final_state_masses, backend="jax"),
        NumpyUniformRNG(seed=0),
    ),
}
for generator, _ in generators.values():
    generator.show_progress = False

| `backend` | Extra | Use of the {class}`.RealNumberGenerator` |
| --- | --- | --- |
| `"numpy"` (default) | `phsp` | {class}`.NumpyUniformRNG` is used directly, any other type only provides a seed |
| `"tensorflow"` | `phsp,tf` | {class}`.TFUniformRealNumberGenerator` is used directly, any other type only provides a seed |
| `"jax"` | `phsp,jax` | any type only provides a seed |

The installation of these extras is described on {doc}`/install`.

## Generating a sample

Every backend is used through the same {meth}`~.TFPhaseSpaceGenerator.generate` method. TensorFlow and JAX compile functions the first time they are called, so we generate two samples per backend and record the time of each. These timings only give an impression for this three-body decay on the machine that builds this page.

In [ ]:
n_events = 100_000
samples = {}
timings = {}
for name, (generator, rng) in generators.items():
    start = time.perf_counter()
    samples[name] = generator.generate(n_events, rng)
    first_call = time.perf_counter() - start
    start = time.perf_counter()
    generator.generate(n_events, rng)
    timings[name] = first_call, time.perf_counter() - start

In [ ]:
pd.DataFrame(
    {
        "Array type": [type(sample["p0"]).__name__ for sample in samples.values()],
        "Shape": [sample["p0"].shape for sample in samples.values()],
        "First call [s]": [round(first, 3) for first, _ in timings.values()],
        "Second call [s]": [round(second, 3) for _, second in timings.values()],
    },
    index=list(samples),
)

## Comparing distributions

The events of different backends are not identical, not even with the same seed, but they should follow the same distribution. A uniform phase space sample shows up as a flat Dalitz plot within the kinematic boundary, and the invariant mass projections should agree between backends.

In [ ]:
def invariant_mass_squared(p: np.ndarray) -> np.ndarray:
    return p[:, 0] ** 2 - np.sum(p[:, 1:] ** 2, axis=1)


fig, axes = plt.subplots(
    ncols=len(samples), figsize=(4 * len(samples), 3.5), sharex=True, sharey=True
)
for ax, (name, sample) in zip(np.atleast_1d(axes), samples.items(), strict=True):
    s01 = invariant_mass_squared(sample["p0"] + sample["p1"])
    s12 = invariant_mass_squared(sample["p1"] + sample["p2"])
    ax.hist2d(s01, s12, bins=60, cmin=1, rasterized=True)
    ax.set_title(name)
    ax.set_xlabel("$m_{01}^2$ [GeV$^2$]")
np.atleast_1d(axes)[0].set_ylabel("$m_{12}^2$ [GeV$^2$]")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, sample in samples.items():
    m01 = np.sqrt(invariant_mass_squared(sample["p0"] + sample["p1"]))
    ax.hist(m01, bins=60, histtype="step", label=name)
ax.set_xlabel("$m_{01}$ [GeV]")
ax.set_ylabel("Events")
ax.legend()
fig.tight_layout()
plt.show()

## Four-body decays

A three-body decay has two independent kinematic variables, so its phase space fits in a single Dalitz plot. A four-body decay has five, and no single plot shows them all. A common choice is the set of [Cabibbo–Maksymowicz variables](https://doi.org/10.1103/PhysRev.137.B438), which split the decay into two pairs of particles:

- the invariant masses $m_{01}$ and $m_{23}$ of the two pairs,
- the helicity angles $\theta_{01}$ and $\theta_{23}$ of particles $0$ and $2$ in the rest frames of their pairs,
- the angle $\phi$ between the two decay planes.

For a uniform phase space sample, $\cos\theta_{01}$, $\cos\theta_{23}$ and $\phi$ are distributed uniformly, while the two masses are not. As an example, we generate $D^0 \to K^-\pi^+\pi^+\pi^-$ with each backend.

In [ ]:
four_body_masses = {0: 0.4937, 1: 0.1396, 2: 0.1396, 3: 0.1396}
four_body_samples = {}
for name, (_, rng) in generators.items():
    generator = TFPhaseSpaceGenerator(1.8648, four_body_masses, backend=name)
    generator.show_progress = False
    four_body_samples[name] = generator.generate(n_events, rng)

In [ ]:
def boost_to_rest_frame(p: np.ndarray, frame: np.ndarray) -> np.ndarray:
    beta = frame[:, 1:] / frame[:, :1]
    beta_squared = np.sum(beta**2, axis=1, keepdims=True)
    gamma = 1 / np.sqrt(1 - beta_squared)
    beta_dot_p = np.sum(beta * p[:, 1:], axis=1, keepdims=True)
    energy = gamma * (p[:, :1] - beta_dot_p)
    momentum = (
        p[:, 1:] + ((gamma - 1) * beta_dot_p / beta_squared - gamma * p[:, :1]) * beta
    )
    return np.concatenate([energy, momentum], axis=1)


def unit(v: np.ndarray) -> np.ndarray:
    return v / np.linalg.norm(v, axis=1, keepdims=True)


def cabibbo_maksymowicz(sample: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    p0, p1, p2, p3 = (sample[f"p{i}"] for i in range(4))
    p01 = p0 + p1
    p23 = p2 + p3
    z = unit(p01[:, 1:])
    n01 = unit(np.cross(p0[:, 1:], p1[:, 1:]))
    n23 = unit(np.cross(p2[:, 1:], p3[:, 1:]))
    sin_phi = np.sum(np.cross(n01, n23) * z, axis=1)
    cos_phi = np.sum(n01 * n23, axis=1)
    return {
        "$m_{01}$ [GeV]": np.sqrt(invariant_mass_squared(p01)),
        "$m_{23}$ [GeV]": np.sqrt(invariant_mass_squared(p23)),
        R"$\cos\theta_{01}$": np.sum(
            unit(boost_to_rest_frame(p0, p01)[:, 1:]) * z, axis=1
        ),
        R"$\cos\theta_{23}$": np.sum(
            unit(boost_to_rest_frame(p2, p23)[:, 1:]) * -z, axis=1
        ),
        R"$\phi$": np.arctan2(sin_phi, cos_phi),
    }


variables = {name: cabibbo_maksymowicz(s) for name, s in four_body_samples.items()}
labels = list(next(iter(variables.values())))
fig, axes = plt.subplots(ncols=len(labels), figsize=(3 * len(labels), 3), sharey=True)
for ax, label in zip(axes, labels, strict=True):
    for name, values in variables.items():
        ax.hist(values[label], bins=50, histtype="step", label=name)
    ax.set_xlabel(label)
axes[0].set_ylabel("Events")
axes[-1].legend()
fig.tight_layout()
plt.show()

The variables are not independent in their ranges: the sum of the two pair masses cannot exceed the mass of the decaying particle. A two-dimensional histogram of $m_{01}$ against $m_{23}$ shows this kinematic boundary, which plays a role similar to the boundary of the Dalitz plot.

In [ ]:
fig, axes = plt.subplots(
    ncols=len(variables), figsize=(4 * len(variables), 3.5), sharex=True, sharey=True
)
for ax, (name, values) in zip(np.atleast_1d(axes), variables.items(), strict=True):
    ax.hist2d(values[labels[0]], values[labels[1]], bins=60, cmin=1, rasterized=True)
    ax.set_title(name)
    ax.set_xlabel(labels[0])
np.atleast_1d(axes)[0].set_ylabel(labels[1])
fig.tight_layout()
plt.show()

## Benchmark

The timings above come from a single sample. For a more systematic comparison, we measure how long each backend takes to generate a weighted sample with {class}`.TFWeightedPhaseSpaceGenerator`, as a function of the sample size and of the number of final state particles. The hit-and-miss step of {class}`.TFPhaseSpaceGenerator` is left out, because it is the same for all backends. Each measurement is preceded by one call of the same size that is not timed, so that compilation does not count, and the fastest of three repetitions is shown.

In [ ]:
from tensorwaves.data import TFWeightedPhaseSpaceGenerator


def measure(backend: str, masses: dict[int, float], size: int) -> float:
    _, rng = generators[backend]
    generator = TFWeightedPhaseSpaceGenerator(5.0, masses, backend=backend)
    generator.generate(size, rng)
    durations = []
    for _ in range(3):
        start = time.perf_counter()
        generator.generate(size, rng)
        durations.append(time.perf_counter() - start)
    return min(durations)


pion_mass = 0.1396
sizes = [10_000, 30_000, 100_000, 300_000]
particle_counts = [3, 4, 5, 6, 7, 8]
timings_by_size = {
    backend: [measure(backend, dict.fromkeys(range(3), pion_mass), n) for n in sizes]
    for backend in generators
}
timings_by_particles = {
    backend: [
        measure(backend, dict.fromkeys(range(k), pion_mass), 100_000)
        for k in particle_counts
    ]
    for backend in generators
}

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(10, 3.5))
for backend in generators:
    ax1.plot(sizes, timings_by_size[backend], marker="o", label=backend)
    ax2.plot(particle_counts, timings_by_particles[backend], marker="o", label=backend)
ax1.set_xscale("log")
ax1.set_xlabel("Number of events")
ax1.set_title("Three-body decay")
ax2.set_xlabel("Number of final state particles")
ax2.set_title("100,000 events")
for ax in (ax1, ax2):
    ax.set_yscale("log")
    ax.set_ylabel("Time [s]")
ax1.legend()
fig.tight_layout()
plt.show()

## Which generator to choose

NumPy is the default choice, because it needs no additional dependencies and no compilation. TensorFlow and JAX pay for compilation on the first call, but are several times faster afterwards, as the [benchmark](#benchmark) shows. They are therefore worth it when generating large or many samples for the same decay. TensorFlow reproduces samples from earlier TensorWaves versions, when combined with a {class}`.TFUniformRealNumberGenerator`. JAX can be useful if the rest of the analysis already uses it.

There are two things to keep in mind:

- The same seed gives a different sample on each backend. Samples are only reproducible within one backend.
- `phasespace` selects its backend per process, and a generator activates its backend each time it generates. Each switch makes `phasespace` recompile, so alternating between backends is slower than the timings above suggest, and generators with different backends should not be used concurrently in threads.